# Demo: from the website to a price estimate

This notebook runs the whole project on a **small sample** (3 pages = 72 cars), in about 20 seconds:

1. **Scrape** the used-car listings of aramisauto.com (`scripts/aramis_scraper.py`)
2. **Clean** the data (`scripts/datacleaning.py`)
3. **Look** at the data
4. **Train** a price model and measure its error (`scripts/models.py`)
5. **Estimate** the price of a car

It needs an internet connection. The scraper waits 5 seconds between two pages, as asked by the `robots.txt` of the website.
To get the whole dataset (about 110 pages), use the commands of the README instead.


In [ ]:
import sys

sys.path.append('..')  # the scripts folder is at the root of the project, one level above this notebook

import matplotlib.pyplot as plt
import pandas as pd

from scripts.aramis_scraper import scrape_listings
from scripts.datacleaning import clean_cars
from scripts.models import predict_price, train_and_evaluate, train_final_model


## 1. Scrape

In [ ]:
rows = scrape_listings(max_pages=3)  # 24 cars per page, 5 seconds between two pages
raw = pd.DataFrame(rows)
print(len(raw), 'cars scraped')
raw.head()


## 2. Clean

`clean_cars` converts the numbers, groups the fuels and gearboxes (`Auto.` becomes `Automatique`), and removes the
incomplete rows, the impossible values and the duplicates.

In [ ]:
cars = clean_cars(raw)
print(len(raw), 'rows scraped,', len(cars), 'kept')
cars.head()


## 3. Look at the data

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for fuel, group in cars.groupby('Carburant'):
    ax.scatter(group['Km'], group['Prix'], label=fuel, alpha=0.7)
ax.set_xlabel('Mileage (km)')
ax.set_ylabel('Price (€)')
ax.set_title('Price and mileage')
ax.legend(title='Fuel')
plt.show()

cars[['Km', 'année', 'Prix']].describe()


## 4. Train a price model

`train_and_evaluate` keeps 20 % of the cars aside and measures the error on them. `baseline_mae` is the error of a
"model" that always answers the average price: ours must do much better.

With only 72 cars the numbers are just an order of magnitude. On the whole dataset the result is much more reliable.

In [ ]:
model, metrics = train_and_evaluate(cars)
print(f"Mean error: {metrics['mae']:.0f} € (always answering the mean price: {metrics['baseline_mae']:.0f} €)")
print(f"R2: {metrics['r2']:.2f} (1 is perfect), tested on {metrics['n_test']} cars")


## 5. Estimate the price of a car

In [ ]:
final_model = train_final_model(cars)  # trained on all the cars
price = predict_price(final_model, 'Peugeot 208', 'Essence', 'Manuelle', km=60_000, year=2019)
print(f'Estimated price: {price:,.0f} €'.replace(',', ' '))


## Next steps

To get the whole dataset and a model you can rely on (from the root of the project):

```bash
python -m scripts.aramis_scraper --all --output data/cars_raw.csv
python -m scripts.datacleaning data/cars_raw.csv -o data/cars_clean.csv
python -m scripts.models data/cars_clean.csv
```
